In [38]:
import pickle
import pandas as pd
from transformers import BertModel, BertTokenizer
import torch
from tqdm import tqdm

In [35]:
import os, sys
import regex as re
import random
from nltk.corpus import wordnet as wn, stopwords
from nltk import pos_tag
_STOP = set(stopwords.words("english"))
_tok_re = re.compile(r"\w+|[^\w\s]", re.UNICODE)
_PENN_TO_WN = {'N': wn.NOUN, 'V': wn.VERB, 'J': wn.ADJ, 'R': wn.ADV}

import torch, random

def mask_augment(input_ids, attention_mask, tokenizer, p=0.2):
        """
        Apply masking augmentation to the input_ids and attention_mask.
        """
        valid_pos = (attention_mask == 1).clone()
        valid_pos[0] = False  # CLS
        if valid_pos.sum() > 2:
                valid_pos[valid_pos.nonzero()[-1]] = False  # SEP

        # Random mask
        rand = torch.rand(input_ids.shape)
        mask = (rand < p) & valid_pos.bool()

        # Apply mask
        input_ids = input_ids.clone()
        input_ids[mask] = tokenizer.mask_token_id
        return input_ids, attention_mask

def _ids_to_tokens(input_ids, tokenizer, attention_mask):
    valid = attention_mask.bool().cpu()
    return tokenizer.convert_ids_to_tokens(input_ids[valid].tolist())

def _tokens_to_ids(tokens, tokenizer, seq_len, device):
    enc = tokenizer(tokens,
                    is_split_into_words=True,
                    padding='max_length',
                    truncation=True,
                    max_length=seq_len,
                    return_tensors='pt')
    return enc['input_ids'][0].to(device), enc['attention_mask'][0].to(device)

def _is_special(tok, tokenizer):
    return tok in {tokenizer.cls_token, tokenizer.sep_token,
                   tokenizer.pad_token, tokenizer.mask_token}

def _find_synonyms(word, pos=None):
    syns = set()
    synsets = wn.synsets(word, pos=pos) if pos else wn.synsets(word)
    for syn in synsets:
        for lemma in syn.lemmas():
            w = lemma.name().replace('_', ' ').lower()
            if w != word.lower():
                syns.add(w)
    return [s for s in syns if _tok_re.fullmatch(s)]

def _join_tokens(tokens):
    out, prev = [], ''
    for curr in tokens:
        if prev and prev[-1].isalnum() and curr.isalnum():
            out.append(' ')
        out.append(curr)
        prev = curr
    return ''.join(out)

def _preserve_case(src, repl):
    """Capitalize replacement if src was capitalized."""
    return repl.capitalize() if src and src[0].isupper() else repl

def delete_augment(input_ids, attention_mask, tokenizer, p=0.2):
    device, seq_len = input_ids.device, input_ids.size(0)
    tokens = _ids_to_tokens(input_ids, tokenizer, attention_mask)

    kept = []
    for t in tokens:
        if _is_special(t, tokenizer):
            kept.append(t)
        else:
            if random.random() > p:
                kept.append(t)

    if len(kept) <= 2:
        kept.append(tokens[1])

    new_ids, new_mask = _tokens_to_ids(kept, tokenizer, seq_len, device)
    return new_ids, new_mask

def synonym_augment(text, alpha=0.1, keep_pos=('N','V','J','R')):
    tokens = _tok_re.findall(text)
    word_positions = [(i, tok) for i, tok in enumerate(tokens) if tok.isalpha() and tok.lower() not in _STOP]
    if not word_positions:
        return text

    _, word_list = zip(*word_positions)
    tags = pos_tag(list(word_list))

    eligible = []
    for (i, tok), (_, tag) in zip(word_positions, tags):
        if tag and tag[0] in keep_pos:
            eligible.append((i, tok, tag[0]))

    # No eligible texts
    if not eligible:
        return text

    k = max(1, int(alpha * len(tokens))) # Per the paper
    chosen = random.sample(eligible, min(k, len(eligible)))

    for idx, orig_tok, penn_first in chosen:
        wn_pos = _PENN_TO_WN.get(penn_first)
        syns = _find_synonyms(orig_tok, wn_pos)
        if syns:
            rep = random.choice(syns)
            tokens[idx] = _preserve_case(orig_tok, rep)

    return _join_tokens(tokens)

PUNCTS = [".", ",", "!", "?", ";", ":"]

def aeda_augment(text, n=2):
    words = text.split()
    for _ in range(n):
        pos = random.randint(0, len(words))
        mark = random.choice(PUNCTS)
        words.insert(pos, mark)
    aug = " ".join(words)
    return aug

def augment_token(input_ids, attention_mask, tokenizer):
        if random.random() < 0.5:
            return mask_augment(input_ids, attention_mask, tokenizer)
        else:
            return delete_augment(input_ids, attention_mask, tokenizer)
        
def augment_text(text):
    temp = synonym_augment(text)
    return aeda_augment(temp)

def augment(text, input_ids, attention_mask, tokenizer):
    if random.random() < 1/3:
        text = augment_text(text)
        return text, None, None
    elif random.random() < 2/3:
        input_ids, attention_mask = augment_token(input_ids, attention_mask, tokenizer)
        return None, input_ids, attention_mask
    else:
        input_ids, attention_mask = mask_augment(input_ids, attention_mask, tokenizer)
        return None, input_ids, attention_mask

In [36]:
def cosine_similarity(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)
    return (a * b).sum(dim=-1)

In [44]:
def check_sim(original, backtranslated, augmented):
    total_sim_o_vs_b = 0
    total_sim_o_vs_a = 0
    count = 0
    for o, b, a in zip(original, backtranslated, augmented):
        tokenized_o = tokenizer(o, return_tensors='pt', padding=True, truncation=True, max_length=512)
        tokenized_b = tokenizer(b, return_tensors='pt', padding=True, truncation=True, max_length=512)
        tokenized_a = tokenizer(a, return_tensors='pt', padding=True, truncation=True, max_length=512)
        o_ids = tokenized_o['input_ids'].to(device)
        o_mask = tokenized_o['attention_mask'].to(device)
        b_ids = tokenized_b['input_ids'].to(device)
        b_mask = tokenized_b['attention_mask'].to(device)
        a_ids = tokenized_a['input_ids'].to(device)
        a_mask = tokenized_a['attention_mask'].to(device)

        o_output = bert(o_ids, attention_mask=o_mask)
        b_output = bert(b_ids, attention_mask=b_mask)
        a_output = bert(a_ids, attention_mask=a_mask)

        o_embedding = o_output.pooler_output
        b_embedding = b_output.pooler_output
        a_embedding = a_output.pooler_output

        sim_b = cosine_similarity(o_embedding, b_embedding)
        sim_a = cosine_similarity(o_embedding, a_embedding)
        # print(f"Cosine Similarity (Original vs Backtranslated): {sim_b.item():.4f}")
        # print(f"Cosine Similarity (Original vs Augmented): {sim_a.item():.4f}")
        # print("-" * 50)
        total_sim_o_vs_b += sim_b.item()
        total_sim_o_vs_a += sim_a.item()
        count += 1
    avg_sim_o_vs_b = total_sim_o_vs_b / count
    avg_sim_o_vs_a = total_sim_o_vs_a / count
    print(f"Average Cosine Similarity (Original vs Backtranslated): {avg_sim_o_vs_b:.4f}")
    print(f"Average Cosine Similarity (Original vs Augmented): {avg_sim_o_vs_a:.4f}")

In [ ]:
bert = BertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
device = torch.device('mps')

/opt/anaconda3/envs/diss/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [13]:
bert.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
with open('de_1.pkl', 'rb') as f:
    de = pickle.load(f)
    backtranslated = list(de.values())

In [31]:
df = pd.read_csv('agnews_train.csv')
original = []
for i, text in de.items():
    original.append(df['Description'].iloc[i])

In [37]:
augmented = [augment_text(text) for text in original]

In [ ]:
check_sim(original, backtranslated, augmented)